# 10 - GRU and CNN-LSTM Architectures for RUL Estimation
### Phase 3: Deep Learning Model Development, Training & Evaluation

This notebook evaluates **Gated Recurrent Units (GRU)** and **Hybrid 1D Conv-LSTM (CNN-LSTM)** models for Remaining Useful Life (RUL) estimation on **NASA C-MAPSS**.

#### Key Objectives:
1. Compare GRU parameter efficiency, training speed, and accuracy against LSTM.
2. Evaluate 1D temporal convolution for localized sensor feature extraction coupled with recurrent dynamics.
3. Compare models on the official test fleet terminal cycle benchmark ($N=100$).


In [ ]:
import sys
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(Path("..").resolve() / "src"))

from deep_learning.dataset_loader import get_cmapss_dataloaders
from deep_learning.gru_model import GRURULModel
from deep_learning.cnn_lstm_model import CNNLSTMRULModel
from deep_learning.evaluate import evaluate_cmapss_rul


## 1. Load Data and Checkpoints for GRU and CNN-LSTM


In [ ]:
loaders_w30 = get_cmapss_dataloaders("FD001", window_length=30, batch_size=256)
in_dim = loaders_w30["num_features"]

gru_model = GRURULModel(input_dim=in_dim, hidden_dim=32, num_layers=1, fc_dim=32)
gru_model.load_state_dict(torch.load("../models/deep_learning/gru/nasa_rul_gru_FD001_w30.pt", map_location="cpu", weights_only=True))
gru_model.eval()

cnn_lstm_model = CNNLSTMRULModel(input_dim=in_dim, conv_filters=32, kernel_size=3, lstm_hidden_dim=32, lstm_layers=1, fc_dim=32)
cnn_lstm_model.load_state_dict(torch.load("../models/deep_learning/cnn_lstm/nasa_rul_cnn_lstm_FD001_w30.pt", map_location="cpu", weights_only=True))
cnn_lstm_model.eval()

with open("../models/deep_learning/gru/nasa_rul_gru_FD001_w30_meta.json", "r") as f:
    gru_meta = json.load(f)
with open("../models/deep_learning/cnn_lstm/nasa_rul_cnn_lstm_FD001_w30_meta.json", "r") as f:
    cnn_meta = json.load(f)

print(f"GRU Parameters:      {gru_meta['size']['total_parameters']:,} ({gru_meta['size']['model_size_kb']} KB)")
print(f"CNN-LSTM Parameters: {cnn_meta['size']['total_parameters']:,} ({cnn_meta['size']['model_size_kb']} KB)")


## 2. Official Terminal Test Fleet Benchmark Comparison


In [ ]:
gru_eval = evaluate_cmapss_rul(
    gru_model, loaders_w30["val_loader"], loaders_w30["test_loader"],
    loaders_w30["val_data"]["y_rul"], loaders_w30["test_data"]["y_rul"],
    loaders_w30["terminal_test_indices"], device="cpu"
)

cnn_eval = evaluate_cmapss_rul(
    cnn_lstm_model, loaders_w30["val_loader"], loaders_w30["test_loader"],
    loaders_w30["val_data"]["y_rul"], loaders_w30["test_data"]["y_rul"],
    loaders_w30["terminal_test_indices"], device="cpu"
)

summary_df = pd.DataFrame([
    {
        "Architecture": "GRU Regressor",
        "Parameters": gru_meta["size"]["total_parameters"],
        "Train Time (s)": gru_meta["train"]["train_duration_sec"],
        "Val RMSE": gru_eval["val_metrics"]["val_rmse"],
        "Test Terminal RMSE": gru_eval["test_terminal_metrics"]["test_terminal_rmse"],
        "Test Terminal MAE": gru_eval["test_terminal_metrics"]["test_terminal_mae"],
        "Test NASA Score": gru_eval["test_terminal_metrics"]["test_terminal_nasa_score"],
    },
    {
        "Architecture": "CNN-LSTM Regressor",
        "Parameters": cnn_meta["size"]["total_parameters"],
        "Train Time (s)": cnn_meta["train"]["train_duration_sec"],
        "Val RMSE": cnn_eval["val_metrics"]["val_rmse"],
        "Test Terminal RMSE": cnn_eval["test_terminal_metrics"]["test_terminal_rmse"],
        "Test Terminal MAE": cnn_eval["test_terminal_metrics"]["test_terminal_mae"],
        "Test NASA Score": cnn_eval["test_terminal_metrics"]["test_terminal_nasa_score"],
    }
])
display(summary_df)


## 3. Comparative Predicted vs Actual RUL


In [ ]:
y_true = gru_eval["y_test_terminal_true"]
y_gru_pred = gru_eval["y_test_terminal_pred"]
y_cnn_pred = cnn_eval["y_test_terminal_pred"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

max_v = max(float(np.max(y_true)), float(np.max(y_gru_pred)), float(np.max(y_cnn_pred))) + 15
ax1.scatter(y_true, y_gru_pred, color="#2ca02c", edgecolors="k", alpha=0.8, s=50)
ax1.plot([0, max_v], [0, max_v], "r--", label="Ideal Identity")
ax1.set_title("GRU: Predicted vs Actual RUL", fontweight="bold")
ax1.set_xlabel("True RUL (Cycles)")
ax1.set_ylabel("Predicted RUL (Cycles)")
ax1.legend()
ax1.grid(True, linestyle=":", alpha=0.6)

ax2.scatter(y_true, y_cnn_pred, color="#d62728", edgecolors="k", alpha=0.8, s=50)
ax2.plot([0, max_v], [0, max_v], "r--", label="Ideal Identity")
ax2.set_title("CNN-LSTM: Predicted vs Actual RUL", fontweight="bold")
ax2.set_xlabel("True RUL (Cycles)")
ax2.legend()
ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()
